## Machine learning pipeline — walkthrough

**Docs:** [`docs/architecture.md`](../docs/architecture.md) (modules, data flow), [`docs/ml_guide.md`](../docs/ml_guide.md) (story + example), [`README.md`](../README.md) (setup, CLI).

### Prerequisites

- **Working directory:** start Jupyter from the **repo root** or from `notebooks/` so the first code cell can set `ROOT` and `sys.path` correctly.
- **Dependencies:** `pip install -r requirements.txt` and `python -m spacy download en_core_web_sm` (spaCy model used in Step 3).
- **LLM (optional):** put `OPENAI_API_KEY` in a `.env` file at the repo root **before Step 9** if you want live API summaries and campaign copy; without it, the step still runs but falls back where the code path expects an API.
- **First run:** Step 1 may download the Kaggle trending CSV if `data/raw/` does not already have it (needs network).

In [1]:
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "src").is_dir():
    ROOT = _here
elif (_here.parent / "src").is_dir():
    ROOT = _here.parent
else:
    raise RuntimeError("Start Jupyter from the repo root or from notebooks/.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repository root:", ROOT)

Repository root: /Users/s0s0rn1/Desktop/Intuit/mail-chimp-code


In [2]:
from dotenv import load_dotenv

from IPython.display import display

import pandas as pd

from src.config.settings import Settings
from src.constants.pipeline_io import (
    TOPIC_INSIGHTS_FILENAME,
    VIDEOS_TEXT_BEFORE_TOPICS_FILENAME,
    VIDEOS_WITH_TOPICS_FILENAME,
)
from src.pipeline.pipeline_run import (
    TrendPipelineContext,
    _step_assign_topics,
    _step_enrich_documents,
    _step_load_dataset,
    _step_prepare_documents,
    _step_save_final_artifacts,
    _step_save_text_prep_checkpoint,
    _step_attach_topic_keywords,
    _step_marketer_insights,
    _step_offline_ranking_evaluation,
    _step_trend_scoring,
)
from src.pipeline.trend_engine import TrendPipelineEngine

load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", 72)

### Configuration and engine

**What:** `Settings` holds paths, `max_rows`, model names, `llm_top_n`, `log_ranking_evaluation`, etc. `TrendPipelineEngine` wires loaders, NLP, BERTopic, scoring, evaluation, and the insight client—same as `main.py`.

**Why `max_rows` here:** smaller subsamples make the notebook faster for teaching; raise or remove the cap for a full pass.


In [3]:
settings = Settings(max_rows=800)
engine = TrendPipelineEngine(settings)

print("max_rows:", settings.max_rows)
print("processed_data_dir:", settings.processed_data_dir)
print("output_dir:", settings.output_dir)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max_rows: 800
processed_data_dir: data/processed
output_dir: outputs


### Pipeline context

**What:** `TrendPipelineContext` is a small bag: `engine`, `videos_df` (one row per video, growing columns as steps run), and `topic_insights_df` (one row per topic, filled from Step 6 onward).

**Why:** each `_step_*` mutates `ctx` in place—same pattern as `run_trend_pipeline()` in code. **Run steps in order**; skipping or reordering breaks downstream assumptions.


In [4]:
ctx = TrendPipelineContext(engine=engine)

#### Step 1: Load dataset

**What:** Bring trending video rows into a pandas `DataFrame`.

**How:** `TrendingDatasetLoader` reads the regional CSV (default: Kaggle `USvideos.csv`), keeps schema columns, and `validate_trending_video_rows` checks types/required fields.

**Why:** Everything downstream assumes clean, validated tabular input; no embeddings or topics yet.

The preview lists every column **ingested for scoring**: `views`, `likes`, `comment_count`, and `trending_date` feed **`TrendScorer.score`** in Step 6 (with `topic` from Step 5). `description` appears only if enabled in `Settings`.


In [5]:
_step_load_dataset(ctx)

v = ctx.videos_df
assert v is not None
print("videos_df shape:", v.shape)
cols = [c for c in ("title", "tags", "views", "likes", "comment_count", "trending_date", "description") if c in v.columns]
display(v[cols].head(2))


Loading dataset from: data/raw/USvideos.csv
videos_df shape: (800, 6)


,title,tags,views,likes,comment_count,trending_date
0,WE WANT TO TALK ABOUT OUR MARRIAGE,SHANtell martin,748374,57527,15954,17.14.11
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),"last week tonight trump presidency|""last week tonight donald trump""|...",2418783,97185,12703,17.14.11


#### Step 2: Build documents

**What:** One text field per video for NLP.

**How:** Concatenate **title**, **tags**, and optionally **description** (`Settings.use_description`) into a column `document`.

**Why:** Clustering and embeddings need a single string per row; this matches how `main.py` builds the document.


In [6]:
_step_prepare_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document") if c in v.columns]
display(v[cols].head(2))

,title,document
0,WE WANT TO TALK ABOUT OUR MARRIAGE,WE WANT TO TALK ABOUT OUR MARRIAGE SHANtell martin
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),The Trump Presidency: Last Week Tonight with John Oliver (HBO) last ...


#### Step 3: spaCy normalization (lemmatized text)

**What:** Turn raw `document` into lighter, more comparable text.

**How:** `SpacyPreprocessor` lemmatizes, drops stopwords/noise, and writes `cleaned_text`.

**Why:** Reduces spelling/noise so sentence embeddings and BERTopic’s bag-of-words statistics are more stable; **embeddings use `cleaned_text`**, not the original title alone.


In [7]:
_step_enrich_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document", "cleaned_text") if c in v.columns]
display(v[cols].head(2))

,title,document,cleaned_text
0,WE WANT TO TALK ABOUT OUR MARRIAGE,WE WANT TO TALK ABOUT OUR MARRIAGE SHANtell martin,want talk marriage shantell martin
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),The Trump Presidency: Last Week Tonight with John Oliver (HBO) last ...,trump presidency week tonight john oliver hbo week tonight trump pre...


#### Step 4: Save text-prep checkpoint (before topics)

**What:** Persist the video table **after** text prep, **before** expensive embedding/topic work.

**How:** Writes `videos_text_before_topics.csv` under `processed_data_dir`.

**Why:** Lets you **reuse** `document` / `cleaned_text` without re-running spaCy, and makes runs **auditable** (diff/checkpoint).


In [8]:
_step_save_text_prep_checkpoint(ctx)

ck = Path(settings.processed_data_dir) / VIDEOS_TEXT_BEFORE_TOPICS_FILENAME
print("Checkpoint:", ck.resolve(), "exists:", ck.is_file())

Checkpoint: /Users/s0s0rn1/Desktop/Intuit/mail-chimp-code/data/processed/videos_text_before_topics.csv exists: True


#### Step 5: Embeddings + topic assignment

**What:** Assign each video to a **topic cluster** (integer id) and a confidence.

**How:** `SentenceTransformer` encodes `cleaned_text` into vectors; **BERTopic** (`TopicModeler.fit_transform`) clusters in embedding space and fits topic representations. Rows get `topic` and `topic_confidence`. **−1** = outlier/noise bucket (excluded from topic-level scoring later).

**Why:** Groups “similar” videos for aggregate trends; **clustering follows embeddings**; keyword statistics inside BERTopic still use the text side of the pipeline.


In [9]:
_step_assign_topics(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "topic", "topic_confidence") if c in v.columns]
display(v[cols].head(5))

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

2026-04-18 15:31:59,295 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-18 15:32:05,247 - BERTopic - Dimensionality - Completed ✓
2026-04-18 15:32:05,248 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-18 15:32:05,285 - BERTopic - Cluster - Completed ✓
2026-04-18 15:32:05,289 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-18 15:32:05,318 - BERTopic - Representation - Completed ✓


,title,topic,topic_confidence
0,WE WANT TO TALK ABOUT OUR MARRIAGE,0,0.589704
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),0,0.932835
2,"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",5,0.406391
3,Nickelback Lyrics: Real or Fake?,-1,0.017171
4,I Dare You: GOING BALD!?,0,0.665666


#### Step 6: Trend scoring

**What:** One row per **topic** with engagement and a **ranking score**.

**How:** `TrendScorer.score` aggregates videos by `topic`, drops `topic == -1`, computes **volume**, **momentum** (latest vs prior day counts), blended **engagement**, then min–max norms and a weighted **`trend_score`** (see `TrendScorer` / `ml_guide`).

**Why:** The dashboard and LLM slice use topic-level signals; this is **heuristic**, not supervised learning-to-rank.


In [10]:
_step_trend_scoring(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = [
    "topic",
    "volume",
    "trend_score",
    "momentum",
    "avg_views",
    "avg_likes",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,volume,trend_score,momentum,avg_views,avg_likes
0,0,95,0.654741,0.000000,942031.863158,25172.347368
1,8,35,0.607039,0.666667,999855.428571,57221.800000
2,1,59,0.529681,0.000000,878026.084746,19894.644068
3,3,58,0.489415,0.117647,581148.327586,13334.448276
4,2,59,0.451197,-0.142857,531101.355932,22594.101695


#### Step 7: Offline ranking evaluation

**What:** Optional **console-only** metric: proxy **NDCG@K** (no human labels).

**How:** Compares ordering by **`trend_score`** to an “ideal” ordering by **`volume`** on the top-**K** rows (K = `llm_top_n`, capped by table length). Toggle with **`log_ranking_evaluation`** on `Settings`.

**Why:** Cheap **sanity check** that the blended score is not wildly misaligned with raw popularity on the same slice—not accuracy, not business KPIs. **How to read:** 1.0 = same order as sorting by `volume` on that slice; lower = more reordering.


In [11]:
_step_offline_ranking_evaluation(ctx)


Ranking evaluation (before LLM enrichment)
------------------------------------------------------------
Proxy NDCG (trend_score order vs ideal by gain column)
  Interpretation: 1.0 = same ordering as sorting by gain; <1.0 = ranker reorders vs pure gain.
  k: 5.0
  gain_col: volume
  score_col: trend_score
  dcg@k (score order): 1.8634272337724511
  idcg@k (ideal by gain): 2.0355146117063327
  ndcg@k (proxy): 0.9154575570500946
------------------------------------------------------------


#### Step 8: Attach topic keywords

**What:** Human-readable **labels and keyword lists** per topic from the **fitted** BERTopic model.

**How:** `add_topic_keyword_columns` fills `topic_keywords`, **`dominant_topic_keywords`**, and **`topic_label`** (short string from the first keywords).

**Why:** Step 9 needs lexical context for taxonomy/coherence checks, display names, and the LLM prompt; **dominant** keywords are what feed classification and the API text (see `enrich_topic_insights_marketer_fields`).


In [12]:
_step_attach_topic_keywords(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = ["topic", "topic_label", "topic_keywords", "dominant_topic_keywords"]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,topic_label,topic_keywords,dominant_topic_keywords
0,0,"late, trump, live, funny, talk","[late, trump, live, funny, talk, jeff, rosie, norton]","[late, trump, live, funny, jeff]"
1,8,"camila, harmony, dashboard, cabello, bass","[camila, harmony, dashboard, cabello, bass, 5th, lele, ally]","[camila, harmony, dashboard, cabello, bass]"
2,1,"iphone, react, face, life, pixel","[iphone, react, face, life, pixel, review, apple, dream]","[iphone, face, pixel, review, apple]"
3,3,"vehicle, jeep, car, mma, wwe","[vehicle, jeep, car, mma, wwe, carmax, crash, tony]","[vehicle, jeep, car, mma, wwe]"
4,2,"makeup, beauty, tan, fish, fashion","[makeup, beauty, tan, fish, fashion, morphe, dry, fei]","[makeup, beauty, tan, fish, fashion]"


#### Step 9: Marketer insights

**What:** Enrich each topic with **taxonomy**, **coherence**, sample titles, optional **LLM** JSON (`summary`, `campaign_copy`, …) for the first **`llm_top_n`** rows (by dataframe index).

**How:** Rule-based checks (`topic_is_coherent`, `classify_trend_type`, `TopicNamer`); if coherent and safe path, **`InsightGenerator.generate_insight`** calls OpenAI with `TREND_INSIGHT_PROMPT_TEMPLATE`; otherwise canned copy.

**Why:** Turns clusters into **marketer-facing** text; the code cell’s **truncated prompt** preview matches `InsightGenerator` for the top **`trend_score`** row (API may still skip incoherent rows).


In [13]:
_step_marketer_insights(ctx)

from src.constants.llm_prompts import TREND_INSIGHT_PROMPT_TEMPLATE

t = ctx.topic_insights_df
assert t is not None
print("topic_insights_df shape:", t.shape)
cols = [
    "topic",
    "topic_label",
    "trend_score",
    "volume",
    "momentum",
    "avg_views",
    "avg_likes",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(8))

row0 = t.iloc[0]
llm_input = TREND_INSIGHT_PROMPT_TEMPLATE.format(
    topic_keywords=row0["dominant_topic_keywords"],
    sample_titles=row0["sample_titles"],
    trend_type=row0["trend_type"],
    avg_views=int(row0["avg_views"]),
    avg_likes=int(row0["avg_likes"]),
    momentum=f"{row0['momentum']:.2f}",
)
print()
print("Sample LLM input (top `trend_score` row; truncated)")
print("=" * 60)
_max = 2400
print(llm_input[:_max] + ("" if len(llm_input) <= _max else "\n…"))
if len(llm_input) > _max:
    print(f"(truncated; full string is {len(llm_input)} chars — see TREND_INSIGHT_PROMPT_TEMPLATE in src/constants/llm_prompts.py)")

topic_insights_df shape: (16, 23)


,topic,topic_label,trend_score,volume,momentum,avg_views,avg_likes
0,0,"late, trump, live, funny, talk",0.654741,95,0.000000,9.420319e+05,25172.347368
1,8,"camila, harmony, dashboard, cabello, bass",0.607039,35,0.666667,9.998554e+05,57221.800000
2,1,"iphone, react, face, life, pixel",0.529681,59,0.000000,8.780261e+05,19894.644068
3,3,"vehicle, jeep, car, mma, wwe",0.489415,58,0.117647,5.811483e+05,13334.448276
4,2,"makeup, beauty, tan, fish, fashion",0.451197,59,-0.142857,5.311014e+05,22594.101695
5,12,"christmas, amazon, advert, lewis, anastasia",0.432050,22,0.400000,1.628780e+06,33858.090909
6,6,"nail, cream, cheese, burrito, gadget",0.417260,44,-0.071429,6.085193e+05,26849.159091
7,5,"batman, movie, trailer, league, justice",0.411499,47,-0.133333,1.215079e+06,42544.212766



Sample LLM input (top `trend_score` row; truncated)

You generate grounded marketing insights for a Mailchimp-style trend engine.

Return ONLY valid JSON with these keys:
- summary
- campaign_angle
- suggested_subject
- email_hook
- marketing_safe

Inputs:
- topic_keywords: ['late', 'trump', 'live', 'funny', 'jeff']
- sample_titles: ['WE WANT TO TALK ABOUT OUR MARRIAGE', 'The Trump Presidency: Last Week Tonight with John Oliver (HBO)', 'I Dare You: GOING BALD!?']
- trend_type: general
- avg_views: 942031
- avg_likes: 25172
- momentum: 0.00

Rules:
1. Use only the provided inputs.
2. If a claim cannot be directly supported by topic_keywords or sample_titles, do not include it.
3. Do not invent specific shows, products, events, controversies, relationships, or storylines.
4. Treat names as context, not the main theme unless clearly dominant across multiple keywords or multiple sample_titles.
5. Do not force categories such as tech, business, politics, or social impact unless clearly sup

#### Step 10: Save final outputs

**What:** Persist the **final** video-level and topic-level tables for the app and reuse.

**How:** `save_final_artifacts` validates rows and writes **`videos_with_topics.csv`** and **`topic_insights.csv`** under `output_dir` (paths in `src/constants/pipeline_io.py`).

**Why:** **`streamlit run app.py`** and `python -m src.evaluation` expect these paths; reproducible handoff from batch run to UI.


In [14]:
_step_save_final_artifacts(ctx)

out = Path(settings.output_dir)
vwt = out / VIDEOS_WITH_TOPICS_FILENAME
tic = out / TOPIC_INSIGHTS_FILENAME
print("videos_with_topics:", vwt.resolve(), vwt.is_file())
print("topic_insights:", tic.resolve(), tic.is_file())

videos_with_topics_df = ctx.videos_df
topic_insights_df = ctx.topic_insights_df

Saved final outputs to: /Users/s0s0rn1/Desktop/Intuit/mail-chimp-code/outputs
videos_with_topics: /Users/s0s0rn1/Desktop/Intuit/mail-chimp-code/outputs/videos_with_topics.csv True
topic_insights: /Users/s0s0rn1/Desktop/Intuit/mail-chimp-code/outputs/topic_insights.csv True


### Final output — top trends (pretty-print)

**What:** Same **pretty-print** loop as `main.py`: walk **`topic_insights_df`** in **`trend_score`** order and print scores plus **`summary`** / **`campaign_copy`**.

**Why:** Quick **human-readable** demo of the end product without opening Streamlit. Does not re-run the pipeline.


In [15]:
assert videos_with_topics_df is not None and topic_insights_df is not None

top_n = settings.top_n_topics_to_show

for _, row in topic_insights_df.head(top_n).iterrows():
    suggestion = row["campaign_copy"]
    print("\n" + "=" * 60)
    print(f"Trend: {row['topic_label']}")
    print(f"Trend Score: {row['trend_score']:.2f}")
    print(f"Volume: {int(row['volume'])}")
    print(f"Avg Views: {int(row['avg_views']):,}")
    print(f"Avg Likes: {int(row['avg_likes']):,}")
    print(f"Momentum: {row['momentum']:.2f}")
    print("\nSummary:")
    print(row["summary"])
    print("\nCampaign copy:")
    print(f"  Suggested Subject: {suggestion['suggested_subject']}")
    print(f"  Campaign Angle: {suggestion['campaign_angle']}")
    print(f"  Email Hook: {suggestion['email_hook']}")


Trend: late, trump, live, funny, talk
Trend Score: 0.65
Volume: 95
Avg Views: 942,031
Avg Likes: 25,172
Momentum: 0.00

Summary:
This cluster highlights popular humorous and late-night style content with references to comedy, real-life topics, and engaging personalities. The content attracts significant views and likes, indicating steady interest in entertainment that blends current events and personal stories.

Campaign copy:
  Suggested Subject: Late-Night Comedy and Real Talks Trend
  Campaign Angle: Leverage humor and relatable storytelling in live or recorded formats to engage audiences looking for entertaining and topical content. Utilize email marketing to tease behind-the-scenes insights or upcoming live events that blend comedy with real-life discussions.
  Email Hook: Discover how humor and honesty are winning over audiences in late-night style content - get your campaign in sync with this steady favorite!

Trend: camila, harmony, dashboard, cabello, bass
Trend Score: 0.61
V